# Clasificación de Noticias

## Data Preparation

In [3]:
from gensim.models import KeyedVectors
import os

w2v_model = KeyedVectors.load('../Representacion_del_lenguaje/embeddings/NoContext/w2v_sg.kv', mmap='r')
print(w2v_model)

KeyedVectors<vector_size=300, 16863 keys>


Para modelos de Deep Learning con redes neuronales, necesitamos una representación diferente a la usada en Shallow Learning. Mientras que para modelos clásicos (Regresión Logística, SVM) promediamos los vectores Word2Vec de todas las palabras en un solo vector de 300 dimensiones, para CNNs necesitamos tener en cuanta la secuencia de palabras en el texto. Cada texto se representa como una matriz 3D donde cada palabra mantiene su posición y su vector Word2Vec de 300 dimensiones.

In [4]:
import pandas as pd

processed_texts = pd.read_parquet('../Representacion_del_lenguaje/processed_data/preprocNoContext_embeddings.parquet')
processed_texts = processed_texts['ncEmbedText']
processed_texts.head()

0    the jobs market continues show signs weakness ...
1    new data from the department from work and pen...
2    asking for workplace accommodations often easi...
3    apple has announced major expansion its renewa...
4    the advent artificial intelligence less than t...
Name: ncEmbedText, dtype: object

Transformamos cada texto en una matriz 2D de dimensiones (MAX_LEN, 300). Para cada texto tokenizamos las palabras; para cada posición hasta MAX_LEN=250, colocamos el vector Word2Vec de 300 dimensiones de esa palabra; si la palabra no está en el vocabulario o el texto es más corto que MAX_LEN, rellenamos con ceros (padding). El resultado final es una matriz 3D de dimensiones (num_textos, 250, 300) donde cada "paso temporal" corresponde a una palabra en su posición original. Esta representación secuencial permite que la CNN capture patrones locales como bigramas o trigrams (kernel_size) sirven para entender el contexto y significado del texto.

In [5]:
import numpy as np
import nltk

MAX_LEN = 250
EMBEDDING_DIM = 300

def crear_secuencias_vectores(texts, modelo_w2v, max_len):
    textos_tokenizados = [nltk.word_tokenize(texto.lower()) for texto in texts]
    num_textos = len(textos_tokenizados)
    # Matriz de ceros
    data_matrix = np.zeros((num_textos, max_len, EMBEDDING_DIM))
    
    for i, texto in enumerate(textos_tokenizados):
        # Para cada palabra en el texto
        for j, palabra in enumerate(texto):
            if j >= max_len:
                break
            
            # Si la palabra existe ponemos su vector
            if palabra in modelo_w2v:
                data_matrix[i, j, :] = modelo_w2v[palabra]
            # Si no existe, se queda en ceros
            
    return data_matrix

In [6]:
X_w2v = crear_secuencias_vectores(processed_texts, w2v_model, MAX_LEN)

print(X_w2v.shape)

(5160, 250, 300)


Codificamos las etiquetas de texto a números (0-4) usando LabelEncoder. Dividimos el dataset en training (80%) y test (20%) con estratificación. A diferencia de Shallow Learning donde teníamos un conjunto de validación separado, aquí usaremos Cross-Validation en el entrenamiento. Los datos de entrenamiento tienen forma (4128, 250, 300): 4128 textos, cada uno con 250 palabras de 300 dimensiones.

In [7]:
print(X_w2v[0])

[[-0.10494587  0.00571608 -0.02699127 ... -0.06973305 -0.11877021
  -0.06472845]
 [-0.16664293  0.3422474   0.37715384 ... -0.19503252  0.08127215
  -0.37072948]
 [ 0.21194062  0.02457219 -0.34651375 ...  0.03259023 -0.29865226
  -0.21168469]
 ...
 [-0.09888955 -0.0249728   0.14641783 ...  0.01026709  0.01079688
   0.287429  ]
 [ 0.25404429  0.23451392  0.52566278 ...  0.24864739 -0.0508902
   0.38398963]
 [ 0.20846635  0.04844557  0.3349756  ... -0.13310026 -0.08682542
   0.1327893 ]]


In [8]:
topics = pd.read_csv("../../data/definitivos/INDEX_ALL_scrapped_filtrado.csv")
y = topics["topic"].values

print("Total de textos:", y.size)
print("\nDistribución de textos por topic:")
print(topics["topic"].value_counts().sort_index())

Total de textos: 5160

Distribución de textos por topic:
topic
Business Growth and Cloud Infrastructure in the AI Industry    1539
Financial and Market News and Corporate Sales                  2257
Informal / Conversational Lenguaje                              152
Quantum Computing and Military Technology                       101
Stock Market and Trading                                       1111
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Codificar las etiquetas de texto a números (0, 1, 2, 3, 4)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Clases codificadas:")
for i, clase in enumerate(label_encoder.classes_):
    print(f"{i}: {clase}")

xw2v_train_global, xw2v_test_global, y_train_global, y_test_global = train_test_split(X_w2v, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("\nTextos (variable predictoria):\n", "Training", xw2v_train_global.shape, "Test", xw2v_test_global.shape)
print("Topics (variable a predecir):\n", "Training", y_train_global.shape, "Test", y_test_global.shape)

Clases codificadas:
0: Business Growth and Cloud Infrastructure in the AI Industry
1: Financial and Market News and Corporate Sales
2: Informal / Conversational Lenguaje
3: Quantum Computing and Military Technology
4: Stock Market and Trading

Textos (variable predictoria):
 Training (4128, 250, 300) Test (1032, 250, 300)
Topics (variable a predecir):
 Training (4128,) Test (1032,)

Textos (variable predictoria):
 Training (4128, 250, 300) Test (1032, 250, 300)
Topics (variable a predecir):
 Training (4128,) Test (1032,)


### Pretrained *Word2Vec*

Para comparar resultados, vamos a utilizar un modelo Word2Vec preentrenado por Google en 3 millones de noticias de Google News (aproximadamente 100 mil millones de palabras). Este modelo contiene vectores de 300 dimensiones para 3 millones de palabras y frases, entrenado con un corpus masivo y diverso. La ventaja de usar embeddings preentrenados es que capturan conocimiento lingüístico general aprendido de grandes volúmenes de texto.

In [10]:
w2v_pretrained_model = KeyedVectors.load_word2vec_format('../models/GoogleNews-vectors-negative300.bin.gz', binary=True)
print(w2v_pretrained_model)

KeyedVectors<vector_size=300, 3000000 keys>


Esta representación alternativa nos permitirá evaluar en el entrenamiento de la CNN si los embeddings genéricos de alta calidad superan a los embeddings especializados pero entrenados con menos datos.

In [11]:
X_w2v_pretrained = crear_secuencias_vectores(processed_texts, w2v_pretrained_model, MAX_LEN)
print(X_w2v_pretrained.shape)

(5160, 250, 300)


In [12]:
print(X_w2v_pretrained[0])

[[ 0.08007812  0.10498047  0.04980469 ...  0.00366211  0.04760742
  -0.06884766]
 [ 0.14746094 -0.04638672  0.18652344 ... -0.11035156  0.14941406
  -0.29882812]
 [-0.15625    -0.08789062 -0.22949219 ...  0.14160156  0.04736328
  -0.0135498 ]
 ...
 [ 0.01831055 -0.11816406  0.0402832  ...  0.05957031 -0.01519775
  -0.03710938]
 [-0.0057373   0.06298828  0.05200195 ...  0.01257324 -0.00878906
  -0.00708008]
 [-0.07177734 -0.11035156  0.00823975 ...  0.02941895 -0.13964844
   0.125     ]]


In [13]:
from sklearn.model_selection import train_test_split

xw2v_train_pretrained, xw2v_test_pretrained = train_test_split(X_w2v_pretrained, test_size=0.2, random_state=42, stratify=y_encoded)

print("\nTextos (variable predictoria):\n", "Training", xw2v_train_pretrained.shape, "Test", xw2v_test_pretrained.shape)


Textos (variable predictoria):
 Training (4128, 250, 300) Test (1032, 250, 300)


## Convolutional Neural Network

Implementamos una Red Neuronal Convolucional (CNN) para clasificación de texto. La red sigue la siguiente arquitectura:

- Capa Conv1D con 128 filtros y kernel_size=5, que detecta patrones de 5 palabras consecutivas
- GlobalMaxPooling1D que extrae la característica más importante de cada filtro, reduciendo dimensionalidad
- Capa densa de 64 neuronas que combina los patrones detectados
- Dropout del 50% para regularización y prevenir overfitting
- Capa de salida con 5 neuronas y softmax para clasificación multiclase.

Usamos *sparse_categorical_crossentropy* como función de pérdida (apropiada para etiquetas codificadas como enteros) y Adam como optimizador.

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Conv1D, GlobalMaxPooling1D
from keras.optimizers import Adam

def modelo_cnn(input_shape, num_filtros=128, kernel_size=5, lr=0.001, dropout=0.5):
    model = Sequential()

    # Capa de entrada explícita
    model.add(Input(shape=input_shape))
    
    # Bloque convolucional
    # Mirar grupos de 5 palabras
    model.add(Conv1D(filters=num_filtros, 
                     kernel_size=kernel_size, 
                     activation='relu'))
    
    # Reduce la dimensionalidad quedándose con el valor más alto (el rasgo más fuerte)
    model.add(GlobalMaxPooling1D())

    # Bloque denso (Clasificación)
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(dropout))

    # Salida
    model.add(Dense(5, activation='softmax'))

    # Compilación
    optimizer = Adam(learning_rate=lr)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])

    return model

Entrenamos la CNN usando **5-Fold Cross-Validation**. En cada fold dividimos los datos de entrenamiento en 80% para entrenar y 20% para validar ese fold, creamos un modelo CNN nuevo, entrenamos 5 epochs con batches de 32 ejemplos, y evaluamos el accuracy en el conjunto de validación de ese fold. El *accuracy promedio* y su desviación estándar indican el rendimiento esperado del modelo en datos no vistos y la estabilidad del modelo (baja desviación = resultados consistentes).

In [29]:
from sklearn.model_selection import KFold
import numpy as np

# Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_accuracy = []

# Dimensiones de entrada para la CNN
input_shape = (xw2v_train_global.shape[1], xw2v_train_global.shape[2])

print(f"Iniciando CV con input shape: {input_shape}...")

fold_idx = 1
for train_index, val_index in kf.split(xw2v_train_global):
    print(f"\n--- Fold {fold_idx}/5 ---")
    
    # Split de datos para este Fold
    X_train_f, X_val_f = xw2v_train_global[train_index], xw2v_train_global[val_index]
    y_train_f, y_val_f = y_train_global[train_index], y_train_global[val_index]
    
    # Crear modelo nuevo
    model = modelo_cnn(input_shape, num_filtros=128, kernel_size=5, dropout=0.5)
    
    # Entrenar
    # epochs bajas en CV para no tardar una eternidad probando
    model.fit(X_train_f, y_train_f, 
              validation_data=(X_val_f, y_val_f),
              epochs=5, 
              batch_size=32, 
              verbose=1)
    
    # Evaluar
    loss, accuracy = model.evaluate(X_val_f, y_val_f, verbose=0)
    resultados_accuracy.append(accuracy)
    print(f"Accuracy del Fold: {accuracy:.4f}")
    
    fold_idx += 1

print("\n" + "="*30)
print(f"ACCURACY PROMEDIO: {np.mean(resultados_accuracy):.4f} (+/- {np.std(resultados_accuracy):.4f})")

Iniciando CV con input shape: (250, 300)...

--- Fold 1/5 ---
Epoch 1/5
Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.5912 - loss: 1.0264 - val_accuracy: 0.7542 - val_loss: 0.7118
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.5912 - loss: 1.0264 - val_accuracy: 0.7542 - val_loss: 0.7118
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.7723 - loss: 0.6283 - val_accuracy: 0.7966 - val_loss: 0.5507
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.7723 - loss: 0.6283 - val_accuracy: 0.7966 - val_loss: 0.5507
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.8531 - loss: 0.4443 - val_accuracy: 0.8341 - val_loss: 0.4433
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.8531 - loss: 0.4443 - val_accuracy: 0.8341 - val_loss: 0.4433
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9043 - loss: 0.2893 - val_accuracy: 0.8475 - val_loss: 0.4427
Epoch 5/5
104/104 ━━━━━━━━━━━━━━

### Usando vectores de Google News

In [30]:
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_accuracy = []

input_shape = (xw2v_train_pretrained.shape[1], xw2v_train_pretrained.shape[2])

print(f"Iniciando CV con input shape: {input_shape}...")

fold_idx = 1
for train_index, val_index in kf.split(xw2v_train_pretrained):
    print(f"\n--- Fold {fold_idx}/5 ---")
    
    # Split de datos para este Fold
    X_train_f, X_val_f = xw2v_train_pretrained[train_index], xw2v_train_pretrained[val_index]
    y_train_f, y_val_f = y_train_global[train_index], y_train_global[val_index]
    
    # Crear modelo nuevo
    model = modelo_cnn(input_shape, num_filtros=128, kernel_size=5, dropout=0.5)
    
    # Entrenar
    model.fit(X_train_f, y_train_f, 
              validation_data=(X_val_f, y_val_f),
              epochs=5, 
              batch_size=32, 
              verbose=1)
    
    # Evaluar
    loss, accuracy = model.evaluate(X_val_f, y_val_f, verbose=0)
    resultados_accuracy.append(accuracy)
    print(f"Accuracy del Fold: {accuracy:.4f}")
    
    fold_idx += 1

print("\n" + "="*30)
print(f"ACCURACY PROMEDIO: {np.mean(resultados_accuracy):.4f} (+/- {np.std(resultados_accuracy):.4f})")

Iniciando CV con input shape: (250, 300)...

--- Fold 1/5 ---
Epoch 1/5
Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.5812 - loss: 1.0346 - val_accuracy: 0.7470 - val_loss: 0.7176
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.5812 - loss: 1.0346 - val_accuracy: 0.7470 - val_loss: 0.7176
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7665 - loss: 0.6429 - val_accuracy: 0.7942 - val_loss: 0.5663
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7665 - loss: 0.6429 - val_accuracy: 0.7942 - val_loss: 0.5663
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.8540 - loss: 0.4369 - val_accuracy: 0.8354 - val_loss: 0.4606
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.8540 - loss: 0.4369 - val_accuracy: 0.8354 - val_loss: 0.4606
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9031 - loss: 0.2977 - val_accuracy: 0.8499 - val_loss: 0.4130
Epoch 5/5
104/104 ━━━━━━━━━━━━━━

La comparación entre los embeddings Word2Vec entrenados por nosotros y los preentrenados de Google News revela que ningun modelo presenta una mejora significante del accuracy, por lo que el vocabulario especializado en finanzas y el contexto específico del dominio tienen el mismo valor que la cobertura general. La falta de especialización no implica que la calidad general de los embeddings sea mas elevada, por lo que el vocabulario mas amplio y general no es más importante que el dominio específico para este problema.

## Bi-LSTM

Para esta estructura, usamos la API Funcional de Keras en lugar de *Sequential*, porque nos permite hacer la bifurcación del Pooling (calcular Max y Mean en paralelo y luego unirlos) de forma muy limpia.

In [1]:
import tensorflow as tf
from keras.layers import Input, Bidirectional, LSTM, GlobalMaxPooling1D, GlobalAveragePooling1D, Concatenate, Dense, Dropout
from keras.models import Model

def modelo_bilstm_pooling(input_shape, num_filtros=128, lr=0.001, dropout=0.5):
    
    inputs = Input(shape=input_shape)

    # Layer Bi-LSTM (256 neuronas)
    x = Bidirectional(LSTM(num_filtros, return_sequences=True))(inputs)

    # El Bloque de Pooling (Igual que antes)
    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)

    x = Concatenate()([avg_pool, max_pool])

    # Layer densa intermedia (64 neuronas)
    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout)(x)

    # Salida
    outputs = Dense(5, activation='softmax')(x)

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

Entrenamos la Bi-LSTM de la misma forma que con la CNN, una vez mas dividiendi los datos de entrenamiento en 80% para entrenar y 20% para validar en cada fold. Entrenamosel modelo en 5 epochs con batches de 32 ejemplos, y evaluamos el accuracy en el conjunto de validación de ese fold.

In [15]:
from sklearn.model_selection import KFold
import numpy as np

# Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_accuracy_bilstm = []

# Dimensiones de entrada para la Bi-LSTM
input_shape = (xw2v_train_global.shape[1], xw2v_train_global.shape[2])

print(f"Iniciando CV con input shape: {input_shape}...")

fold_idx = 1
for train_index, val_index in kf.split(xw2v_train_global):
    print(f"\n--- Fold {fold_idx}/5 ---")
    
    # Split de datos para este Fold
    X_train_f, X_val_f = xw2v_train_global[train_index], xw2v_train_global[val_index]
    y_train_f, y_val_f = y_train_global[train_index], y_train_global[val_index]
    
    # Crear modelo nuevo
    model = modelo_bilstm_pooling(input_shape, num_filtros=128, dropout=0.5)
    
    # Entrenar
    model.fit(X_train_f, y_train_f, 
              validation_data=(X_val_f, y_val_f),
              epochs=5, 
              batch_size=32, 
              verbose=1)
    
    # Evaluar
    loss, accuracy = model.evaluate(X_val_f, y_val_f, verbose=0)
    resultados_accuracy_bilstm.append(accuracy)
    print(f"Accuracy del Fold: {accuracy:.4f}")
    
    fold_idx += 1

print("\n" + "="*30)
print(f"ACCURACY PROMEDIO: {np.mean(resultados_accuracy_bilstm):.4f} (+/- {np.std(resultados_accuracy_bilstm):.4f})")

Iniciando CV con input shape: (250, 300)...

--- Fold 1/5 ---
Epoch 1/5
Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 25s 209ms/step - accuracy: 0.6233 - loss: 0.9824 - val_accuracy: 0.7022 - val_loss: 0.7563
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 25s 209ms/step - accuracy: 0.6233 - loss: 0.9824 - val_accuracy: 0.7022 - val_loss: 0.7563
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 184ms/step - accuracy: 0.7604 - loss: 0.6812 - val_accuracy: 0.7373 - val_loss: 0.6853
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 184ms/step - accuracy: 0.7604 - loss: 0.6812 - val_accuracy: 0.7373 - val_loss: 0.6853
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 185ms/step - accuracy: 0.7944 - loss: 0.5634 - val_accuracy: 0.7530 - val_loss: 0.6687
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 185ms/step - accuracy: 0.7944 - loss: 0.5634 - val_accuracy: 0.7530 - val_loss: 0.6687
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 20s 189ms/step - accuracy: 0.8292 - loss: 0.4654 - val_accuracy: 0.8414 - val_loss: 0.4485
Epoch 5/5
104/104 

### Usando vectores de Google News

In [16]:
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_accuracy_bilstm_pretrained = []

input_shape = (xw2v_train_pretrained.shape[1], xw2v_train_pretrained.shape[2])

print(f"Iniciando CV con input shape: {input_shape}...")

fold_idx = 1
for train_index, val_index in kf.split(xw2v_train_pretrained):
    print(f"\n--- Fold {fold_idx}/5 ---")
    
    # Split de datos para este Fold
    X_train_f, X_val_f = xw2v_train_pretrained[train_index], xw2v_train_pretrained[val_index]
    y_train_f, y_val_f = y_train_global[train_index], y_train_global[val_index]
    
    # Crear modelo nuevo
    model = modelo_bilstm_pooling(input_shape, num_filtros=128, dropout=0.5)
    
    # Entrenar
    model.fit(X_train_f, y_train_f, 
              validation_data=(X_val_f, y_val_f),
              epochs=5, 
              batch_size=32, 
              verbose=1)
    
    # Evaluar
    loss, accuracy = model.evaluate(X_val_f, y_val_f, verbose=0)
    resultados_accuracy_bilstm_pretrained.append(accuracy)
    print(f"Accuracy del Fold: {accuracy:.4f}")
    
    fold_idx += 1

print("\n" + "="*30)
print(f"ACCURACY PROMEDIO: {np.mean(resultados_accuracy_bilstm_pretrained):.4f} (+/- {np.std(resultados_accuracy_bilstm_pretrained):.4f})")

Iniciando CV con input shape: (250, 300)...

--- Fold 1/5 ---
Epoch 1/5
Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 27s 180ms/step - accuracy: 0.6024 - loss: 0.9946 - val_accuracy: 0.7300 - val_loss: 0.7753
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 27s 180ms/step - accuracy: 0.6024 - loss: 0.9946 - val_accuracy: 0.7300 - val_loss: 0.7753
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 21s 206ms/step - accuracy: 0.7335 - loss: 0.7211 - val_accuracy: 0.7881 - val_loss: 0.6062
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 21s 206ms/step - accuracy: 0.7335 - loss: 0.7211 - val_accuracy: 0.7881 - val_loss: 0.6062
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 181ms/step - accuracy: 0.7689 - loss: 0.6413 - val_accuracy: 0.7627 - val_loss: 0.6128
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 181ms/step - accuracy: 0.7689 - loss: 0.6413 - val_accuracy: 0.7627 - val_loss: 0.6128
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 19s 181ms/step - accuracy: 0.8137 - loss: 0.5402 - val_accuracy: 0.8111 - val_loss: 0.5113
Epoch 5/5
104/104 

La comparación entre CNN y Bi-LSTM revela diferencias importantes. Mientras que la CNN captura patrones locales (g-grams) mediante sliding windows, la Bi-LSTM puede modelar dependencias a largo plazo en ambas direcciones del texto. El rendimiento de la CNN es ligeramente superior al de la Bi-LSTM (Menor tiempo de ejecucion y ligeramente mejores resultados), lo cual indica que los patrones locales de 5 palabras son suficientes para distinguir entre las categorías, siendo más eficiente computacionalmente.